In [1]:
pip install -U langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# Now, any model download will automatically use this token

In [3]:
import pandas as pd
import numpy as np
import torch
import json
import time
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
# Load and Clean Dataset

PATH = "/kaggle/input/datasets/rehanfargose/sc-dataset-1990-2025/SC_Parquet_Dataset_1990-2025.parquet"
df = pd.read_parquet(PATH)
# Filter for non-null and minimum length to ensure quality
df = df[df['full_text'].notna() & (df['full_text'].str.len() > 200)]

In [5]:
# 2. Smart Chunking for BGE-M3 (8k window)
# target ~7500 characters to stay safely within the 8192 token limit
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len
)

all_chunks = []
payloads = []

print(f"Processing {len(df)} judgments into chunks...")

Processing 26515 judgments into chunks...


In [6]:
for _, row in df.iterrows():
    chunks = text_splitter.split_text(row['full_text'])
    
    # Convert list columns to strings for ChromaDB compatibility
    precedents_str = ", ".join(row['precedents']) if isinstance(row['precedents'], list) else str(row['precedents'])
    acts_str = ", ".join(row['acts']) if isinstance(row['acts'], list) else str(row['acts'])
    
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        # Unique ID for every chunk based on the file_id
        chunk_id = f"{row['file_id']}_c{i}"
        
        # Meta-data mapping for advanced filtering
        payloads.append({
            "id": chunk_id,
            "text": chunk,
            "metadata": {
                "year": int(row['year']),
                "case_name": str(row['case_name']),
                "coram": str(row['coram']),
                "decision_date": str(row['decision_date']),
                "acts": acts_str,
                "precedents": precedents_str,
                "neutral_citation": str(row['neutral_citation']),
                "case_no": str(row['case_no']), # Adding Case Number
                "disposal_nature": str(row['disposal_nature']), # Adding Outcome[cite: 1]
            }
        })

In [7]:
# Parallel Embedding Generation on 2x T4
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
# Start multi-process pool for both CUDA devices
pool = model.start_multi_process_pool(target_devices=['cuda:0', 'cuda:1'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
print(f"Generating embeddings for {len(all_chunks)} total chunks...")
start_time = time.time()

Generating embeddings for 737116 total chunks...


In [9]:
# Encode using multi-process pool
embeddings = model.encode(
    all_chunks, 
    pool=pool,              
    batch_size=64,          
    show_progress_bar=True  
)

Chunks:   0%|          | 0/148 [00:00<?, ?it/s]

In [10]:
model.stop_multi_process_pool(pool)
print(f"Completed! Total time: {(time.time() - start_time)/60:.2f} minutes")

Completed! Total time: 58.75 minutes


In [11]:
# 4. Save Outputs for Download
np.save("sc_vectors.npy", embeddings)
with open("sc_payload.json", "w") as f:
    json.dump(payloads, f)

In [12]:
print("Files saved: sc_vectors.npy, sc_payload.json")

Files saved: sc_vectors.npy, sc_payload.json


In [13]:
import shutil

# Create  temp folder to hold files
os.makedirs("legal_index_package", exist_ok=True)

# Move generated files into folder
os.rename("sc_vectors.npy", "legal_index_package/sc_vectors.npy")
os.rename("sc_payload.json", "legal_index_package/sc_payload.json")

# Zip the entire folder
shutil.make_archive("legal_index_package", 'zip', "legal_index_package")

print("Successfully created: legal_index_package.zip")

Successfully created: legal_index_package.zip
